# BigAlpha SQL Model Candidate sql101_hyb_pos_m93_r07_a20_lb760_rt28

FACTOR_SIGN: `1.0`  
FACTOR_THEME: `hybrid`  
MODEL_WEIGHT: `0.93`  
RULE_WEIGHT: `0.07`  
MODEL_ALGO: `ridge`  
RIDGE_ALPHA: `20.0`  
TARGET_HORIZON_DAYS: `1`  
TARGET_TRANSFORM_MODE: `zscore`  
MODEL_FEATURE_SET: `core`  
FACTOR_POSTPROCESS_MODE: `zscore`  
LOOKBACK_DAYS: `760`  
RETRAIN_EVERY_DAYS: `28`  
COMMON_NEUTRALIZE: `False`  
COMMON_NEUTRALIZE_BLEND_ORIGINAL: `0.1`  
Lane: `next_submit_model_weight`  
Reason: single next submission after sql02/model-heavy direction scored better than sql01


In [1]:
"""BigAlpha 2026 AI Factor Mining submission template.

This file is designed for the BigQuant competition environment.
The evaluator imports `main` and calls it with a data source name, start datetime,
and end datetime. The returned DataFrame must contain exactly three columns:
`date`, `instrument`, `factor`.

Core idea:
- Use only the allowed BigAlpha minute bar / order-book data source.
- Aggregate 1-minute data to daily stock-level features with DAI.
- Build interpretable microstructure signals: intraday reversal, order-book imbalance,
  spread pressure, liquidity, volatility and momentum.
- [AI-CORE] Train a rolling Ridge model on historical features to predict next-day
  cross-sectional returns. For each output date, the model only uses observations
  strictly before that date, avoiding look-ahead bias.
"""

from __future__ import annotations

import warnings
from typing import Any, List

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

EPS = 1e-8
DEFAULT_BAR_TABLE = "bigalpha_2026_stock_bar1m"
CANDIDATE_BAR_TABLES = (
    "bigalpha_2026_stock_bar1m",
    "bigalpha_stock_bar1m",
    "cn_stock_bar1m",
    "stock_bar1m",
)
MIN_TRAIN_ROWS = 2_000
MAX_TRAIN_ROWS = 180_000
RETRAIN_EVERY_DAYS = 28
LOOKBACK_DAYS = 760
RANDOM_SEED = 42
FACTOR_SIGN = 1.0
FACTOR_THEME = 'hybrid'
MODEL_WEIGHT = 0.93
RULE_WEIGHT = 0.07
MODEL_ALGO = 'ridge'
RIDGE_ALPHA = 20.0
TARGET_HORIZON_DAYS = 1
TARGET_TRANSFORM_MODE = 'zscore'
MODEL_FEATURE_SET = 'core'
FACTOR_POSTPROCESS_MODE = 'zscore'
COMMON_NEUTRALIZE = False
COMMON_NEUTRALIZE_BLEND_ORIGINAL = 0.1
COMMON_NEUTRALIZE_RIDGE = 1.0


def _resolve_bar_table(data_source: Any) -> str:
    """Resolve the minute-bar table name from the platform input."""
    if isinstance(data_source, str) and data_source.strip():
        return data_source.strip()
    if isinstance(data_source, dict):
        for key in ["bar1m", "stock_bar1m", "minute_bar", "data_source", "table"]:
            value = data_source.get(key)
            if isinstance(value, str) and value.strip():
                return value.strip()
    return DEFAULT_BAR_TABLE


def _candidate_bar_tables(primary: str) -> List[str]:
    candidates = [primary, DEFAULT_BAR_TABLE, *CANDIDATE_BAR_TABLES]
    deduped = []
    seen = set()
    for table in candidates:
        table = str(table).strip()
        if not table or table in seen:
            continue
        seen.add(table)
        deduped.append(table)
    return deduped


def _to_timestamp(value: Any, default: str) -> pd.Timestamp:
    if value is None:
        return pd.Timestamp(default)
    ts = pd.to_datetime(value)
    if pd.isna(ts):
        return pd.Timestamp(default)
    return pd.Timestamp(ts)


def _fmt_filter_datetime(ts: pd.Timestamp, end_of_day: bool = False) -> str:
    ts = pd.Timestamp(ts)
    if end_of_day:
        ts = ts.normalize() + pd.Timedelta(hours=23, minutes=59, seconds=59)
    else:
        ts = ts.normalize()
    return ts.strftime("%Y-%m-%d %H:%M:%S")


def _safe_divide(a: pd.Series, b: pd.Series) -> pd.Series:
    return a.astype(float) / b.astype(float).replace(0, np.nan)


def _cs_zscore(frame: pd.DataFrame, col: str) -> pd.Series:
    """Cross-sectional z-score by date; robust to all-NaN / constant groups."""
    def z(s: pd.Series) -> pd.Series:
        s = pd.to_numeric(s, errors="coerce")
        med = s.median()
        s = s.fillna(med)
        std = s.std(ddof=0)
        if not np.isfinite(std) or std < EPS:
            return pd.Series(0.0, index=s.index)
        return ((s - s.mean()) / (std + EPS)).clip(-5, 5)

    return frame.groupby("date", group_keys=False)[col].apply(z)


def _load_daily_features_from_dai(
    bar_table: str,
    query_start: pd.Timestamp,
    query_end: pd.Timestamp,
) -> pd.DataFrame:
    """Aggregate BigAlpha 1-minute bar and level-1 order-book data to daily features."""
    import dai  # Imported inside the function because it only exists in BigQuant AIStudio.

    start_s = _fmt_filter_datetime(query_start, end_of_day=False)
    end_s = _fmt_filter_datetime(query_end, end_of_day=True)

    errors = []
    for table in _candidate_bar_tables(bar_table):
        for include_book in [True, False]:
            sql = _daily_feature_sql(table, include_book=include_book)
            try:
                data = dai.query(
                    sql,
                    filters={"date": [start_s, end_s]},
                    compression=True,
                ).df()
            except Exception as exc:
                mode = "book" if include_book else "price_volume"
                errors.append(f"{table}/{mode}: {exc}")
                continue
            data = _ensure_daily_columns(data)
            if not data.empty:
                return data
            mode = "book" if include_book else "price_volume"
            errors.append(f"{table}/{mode}: empty result")

    detail = "; ".join(errors[-6:]) if errors else "no query attempts"
    raise RuntimeError(f"Unable to load BigAlpha minute data. Recent attempts: {detail}")


def _daily_feature_sql(bar_table: str, include_book: bool = True) -> str:
    book_fields = ""
    if include_book:
        book_fields = """
            AVG(ask_price1) AS ask_price1_avg,
            AVG(bid_price1) AS bid_price1_avg,
            AVG(ask_volume1) AS ask_volume1_avg,
            AVG(bid_volume1) AS bid_volume1_avg
        """
    selected_fields = f"""
            LAST(pre_close) AS pre_close,
            MAX(high) AS high,
            MIN(low) AS low,
            LAST(close) AS close,
            LAST(volume) AS volume,
            LAST(amount) AS amount,
            AVG(close) AS avg_close{"," if include_book else ""}
            {book_fields}
    """
    return f"""
        SELECT
            date::DATE::DATETIME AS date,
            instrument,
            {selected_fields}
        FROM {bar_table}
        GROUP BY date::DATE, instrument
        ORDER BY date, instrument
    """


def _ensure_daily_columns(daily: pd.DataFrame) -> pd.DataFrame:
    df = daily.copy()
    if df.empty:
        return df
    if "date" not in df.columns or "instrument" not in df.columns:
        raise ValueError("DAI daily data must include date and instrument")

    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["instrument"] = df["instrument"].astype(str)
    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)

    for col in ["close", "pre_close", "high", "low", "volume", "amount", "avg_close"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "close" not in df.columns:
        raise ValueError("DAI daily data must include close")
    if "pre_close" not in df.columns:
        df["pre_close"] = df.groupby("instrument")["close"].shift(1)
    df["pre_close"] = pd.to_numeric(df["pre_close"], errors="coerce").fillna(df["close"])
    if "high" not in df.columns:
        df["high"] = df[["close", "pre_close"]].max(axis=1)
    if "low" not in df.columns:
        df["low"] = df[["close", "pre_close"]].min(axis=1)
    if "volume" not in df.columns:
        df["volume"] = np.nan
    if "amount" not in df.columns:
        df["amount"] = np.nan
    if "avg_close" not in df.columns:
        df["avg_close"] = (df["close"] + df["pre_close"]) / 2.0

    for col in ["ask_price1_avg", "bid_price1_avg"]:
        if col not in df.columns:
            df[col] = df["close"]
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(df["close"])
    for col in ["ask_volume1_avg", "bid_volume1_avg"]:
        if col not in df.columns:
            df[col] = np.nan
        df[col] = pd.to_numeric(df[col], errors="coerce")
    if "minute_count" not in df.columns:
        df["minute_count"] = np.nan

    ordered = [
        "date", "instrument", "pre_close", "high", "low", "close", "volume", "amount",
        "avg_close", "ask_price1_avg", "bid_price1_avg", "ask_volume1_avg",
        "bid_volume1_avg", "minute_count",
    ]
    for col in ordered:
        if col not in df.columns:
            df[col] = np.nan
    return df[ordered]


def _build_daily_features(daily: pd.DataFrame) -> pd.DataFrame:
    """Create predictive daily features from daily aggregated minute data."""
    df = _ensure_daily_columns(daily)
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["instrument"] = df["instrument"].astype(str)
    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)

    numeric_cols = [
        "pre_close", "high", "low", "close", "volume", "amount", "avg_close",
        "ask_price1_avg", "bid_price1_avg", "ask_volume1_avg", "bid_volume1_avg",
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    mid = (df["ask_price1_avg"] + df["bid_price1_avg"]) / 2.0
    vol_sum = df["ask_volume1_avg"] + df["bid_volume1_avg"]

    df["ret_1d"] = _safe_divide(df["close"], df["pre_close"]) - 1.0
    df["range_pct"] = _safe_divide(df["high"] - df["low"], df["pre_close"].abs())
    df["avg_to_close"] = _safe_divide(df["avg_close"], df["close"]) - 1.0
    df["spread_pct"] = _safe_divide(df["ask_price1_avg"] - df["bid_price1_avg"], mid.abs())
    df["order_imbalance"] = _safe_divide(df["bid_volume1_avg"] - df["ask_volume1_avg"], vol_sum.abs())
    df["amount_log"] = np.log1p(df["amount"].clip(lower=0))
    df["volume_log"] = np.log1p(df["volume"].clip(lower=0))
    df["vwap"] = _safe_divide(df["amount"], df["volume"])
    df["close_to_vwap"] = _safe_divide(df["close"], df["vwap"]) - 1.0
    df["illiquidity"] = _safe_divide(df["ret_1d"].abs(), df["amount"].abs() + 1.0)
    range_abs = (df["high"] - df["low"]).abs()
    range_mid = (df["high"] + df["low"]) / 2.0
    df["close_position"] = (_safe_divide(df["close"] - df["low"], range_abs) - 0.5).clip(-0.5, 0.5)
    df["close_to_midrange"] = _safe_divide(df["close"], range_mid.abs()) - 1.0
    df["return_efficiency"] = _safe_divide(df["ret_1d"], df["range_pct"].abs() + EPS).clip(-5.0, 5.0)

    g = df.groupby("instrument", group_keys=False)
    df["mom_5d"] = g["close"].pct_change(5)
    df["mom_20d"] = g["close"].pct_change(20)
    df["mom_60d"] = g["close"].pct_change(60)
    df["vol_5d"] = g["ret_1d"].transform(lambda s: s.rolling(5, min_periods=3).std())
    df["vol_20d"] = g["ret_1d"].transform(lambda s: s.rolling(20, min_periods=5).std())
    df["range_5d"] = g["range_pct"].transform(lambda s: s.rolling(5, min_periods=3).mean())
    df["range_20d"] = g["range_pct"].transform(lambda s: s.rolling(20, min_periods=5).mean())
    df["close_position_5d"] = g["close_position"].transform(lambda s: s.rolling(5, min_periods=3).mean())
    df["imbalance_5d"] = g["order_imbalance"].transform(lambda s: s.rolling(5, min_periods=3).mean())
    df["spread_5d"] = g["spread_pct"].transform(lambda s: s.rolling(5, min_periods=3).mean())
    df["avg_to_close_5d"] = g["avg_to_close"].transform(lambda s: s.rolling(5, min_periods=3).mean())
    df["amount_z20"] = g["amount_log"].transform(
        lambda s: (s - s.rolling(20, min_periods=5).mean()) / (s.rolling(20, min_periods=5).std() + EPS)
    )
    amount_mean_5d = g["amount_log"].transform(lambda s: s.rolling(5, min_periods=3).mean())
    amount_mean_20d = g["amount_log"].transform(lambda s: s.rolling(20, min_periods=5).mean())
    df["amount_trend_5d"] = amount_mean_5d - amount_mean_20d
    df["range_expansion_20d"] = _safe_divide(df["range_pct"], df["range_20d"].abs() + EPS) - 1.0
    df["vol_change_5_20"] = _safe_divide(df["vol_5d"], df["vol_20d"].abs() + EPS) - 1.0
    df["reversal_liquidity"] = -df["ret_1d"] * df["amount_z20"]

    # Target is only used for rolling historical model training.
    horizon = max(1, int(TARGET_HORIZON_DAYS))
    df["target_next"] = g["close"].shift(-horizon) / df["close"] - 1.0
    df["target_label_date"] = g["date"].shift(-horizon)
    df["target_z"] = _target_training_signal(df, "target_next")
    return df.sort_values(["date", "instrument"]).reset_index(drop=True)


def _target_training_signal(df: pd.DataFrame, target_col: str) -> pd.Series:
    """Transform future returns into a robust cross-sectional training label."""
    mode = str(TARGET_TRANSFORM_MODE or "zscore").strip().lower()
    if mode == "rank":
        ranked = df.groupby("date", group_keys=False)[target_col].apply(_rank_to_uniform_score)
        work = df.assign(_target_ranked=ranked)
        return _cs_zscore(work, "_target_ranked")
    if mode == "tanh":
        target_z = _cs_zscore(df, target_col)
        work = df.assign(_target_tanh=np.tanh(target_z / 2.0))
        return _cs_zscore(work, "_target_tanh")
    return _cs_zscore(df, target_col)


def _add_rule_factor(df: pd.DataFrame) -> pd.DataFrame:
    """Build an interpretable non-AI fallback factor."""
    out = df.copy()
    base_cols = [
        "ret_1d",
        "avg_to_close_5d",      # intraday reversal: close below intraday average may rebound
        "imbalance_5d",         # bid-side depth pressure
        "close_to_vwap",        # price-vwap dislocation
        "amount_z20",           # abnormal liquidity participation
        "mom_5d",
        "mom_20d",              # medium-term momentum
        "mom_60d",
        "spread_5d",            # trading friction / crowding risk
        "vol_20d",              # realized risk
        "range_5d",             # intraday instability
        "illiquidity",          # liquidity risk
        "close_position_5d",
        "return_efficiency",
        "amount_trend_5d",
        "range_expansion_20d",
        "vol_change_5_20",
        "reversal_liquidity",
    ]
    for col in base_cols:
        out[col + "_z"] = _cs_zscore(out, col)

    weights = _theme_weights(FACTOR_THEME)
    out["rule_factor"] = 0.0
    for col, weight in weights.items():
        out["rule_factor"] = out["rule_factor"] + weight * out[col + "_z"]
    out["rule_factor"] = _cs_zscore(out, "rule_factor")
    return out


def _theme_weights(theme: str) -> dict:
    """Return rule-factor weights for one low-correlation economic theme."""
    theme = str(theme or "hybrid").strip().lower()
    themes = {
        "hybrid": {
            "avg_to_close_5d": 0.24,
            "imbalance_5d": 0.22,
            "close_to_vwap": -0.12,
            "amount_z20": 0.10,
            "mom_20d": 0.12,
            "spread_5d": -0.16,
            "vol_20d": -0.12,
            "range_5d": -0.10,
            "illiquidity": -0.06,
        },
        "mean_reversion": {
            "avg_to_close_5d": 0.34,
            "close_to_vwap": -0.26,
            "ret_1d": -0.16,
            "mom_5d": -0.12,
            "amount_z20": 0.10,
            "spread_5d": -0.10,
            "vol_20d": -0.08,
        },
        "trend_quality": {
            "mom_20d": 0.28,
            "mom_60d": 0.24,
            "mom_5d": 0.12,
            "amount_z20": 0.10,
            "imbalance_5d": 0.10,
            "vol_20d": -0.14,
            "range_5d": -0.08,
            "spread_5d": -0.08,
        },
        "liquidity_pressure": {
            "imbalance_5d": 0.36,
            "spread_5d": -0.24,
            "illiquidity": -0.16,
            "amount_z20": 0.14,
            "avg_to_close_5d": 0.10,
            "range_5d": -0.08,
        },
        "risk_defensive": {
            "vol_20d": -0.28,
            "range_5d": -0.22,
            "illiquidity": -0.18,
            "spread_5d": -0.14,
            "mom_20d": 0.10,
            "imbalance_5d": 0.08,
        },
        "volume_reversal": {
            "amount_z20": 0.28,
            "ret_1d": -0.20,
            "close_to_vwap": -0.20,
            "avg_to_close_5d": 0.18,
            "mom_5d": -0.10,
            "spread_5d": -0.08,
            "vol_20d": -0.06,
        },
    }
    return themes.get(theme, themes["hybrid"])


def _add_walk_forward_ai_factor(
    df: pd.DataFrame,
    start_dt: pd.Timestamp,
    end_dt: pd.Timestamp,
) -> pd.DataFrame:
    """[AI-CORE] Rolling model training and out-of-sample prediction.

    For each prediction date d, training rows are restricted to date < d. This
    avoids using the target of the prediction date or any future label.
    """
    out = _add_model_feature_columns(df.copy())
    out["ai_model_pred"] = np.nan

    feature_cols = _model_feature_columns(out)

    output_dates = sorted(out.loc[out["date"].between(start_dt, end_dt), "date"].unique())
    if not output_dates:
        return out

    try:
        from sklearn.impute import SimpleImputer
        from sklearn.linear_model import ElasticNet, Ridge
        from sklearn.pipeline import Pipeline
        from sklearn.preprocessing import StandardScaler
    except Exception:
        # BigQuant usually has sklearn. If not, the rule factor still returns a valid submission.
        return out

    model = None
    last_train_date = None
    rng = np.random.default_rng(RANDOM_SEED)

    for d in output_dates:
        d = pd.Timestamp(d)
        need_train = model is None or last_train_date is None or (d - last_train_date).days >= RETRAIN_EVERY_DAYS
        if need_train:
            train_start = d - pd.Timedelta(days=LOOKBACK_DAYS)
            train_mask = (
                (out["date"] < d)
                & (out["date"] >= train_start)
                & (out["target_label_date"] < d)
                & out["target_z"].notna()
            )
            train = out.loc[train_mask, feature_cols + ["target_z"]].replace([np.inf, -np.inf], np.nan)
            train = train.dropna(subset=["target_z"])

            if len(train) >= MIN_TRAIN_ROWS:
                if len(train) > MAX_TRAIN_ROWS:
                    sample_idx = rng.choice(train.index.to_numpy(), size=MAX_TRAIN_ROWS, replace=False)
                    train = train.loc[sample_idx]

                # [AI-CORE] Rolling linear models keep the signal stable and fast
                # under the 3-hour Notebook limit. Ridge is the default; adaptive
                # search can test a sparse ElasticNet variant.
                model = _make_model_pipeline(SimpleImputer, Pipeline, Ridge, StandardScaler, ElasticNet)
                model.fit(train[feature_cols], train["target_z"])
                last_train_date = d

        if model is not None:
            pred_mask = out["date"].eq(d)
            x_pred = out.loc[pred_mask, feature_cols].replace([np.inf, -np.inf], np.nan)
            out.loc[pred_mask, "ai_model_pred"] = model.predict(x_pred)

    out["ai_model_pred"] = _cs_zscore(out, "ai_model_pred")
    return out


def _make_model_pipeline(SimpleImputer, Pipeline, Ridge, StandardScaler, ElasticNet):
    """Create the rolling estimator pipeline for the selected model family."""
    algo = str(MODEL_ALGO or "ridge").strip().lower()
    if algo == "elastic_net":
        estimator = ElasticNet(
            alpha=_elastic_net_alpha(RIDGE_ALPHA),
            l1_ratio=0.15,
            max_iter=600,
            random_state=RANDOM_SEED,
            selection="cyclic",
        )
        step_name = "elastic_net"
    else:
        estimator = Ridge(alpha=RIDGE_ALPHA, random_state=RANDOM_SEED)
        step_name = "ridge"
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (step_name, estimator),
        ]
    )


def _elastic_net_alpha(alpha: float) -> float:
    """Map Ridge-scale alpha to a conservative ElasticNet regularization scale."""
    value = float(alpha)
    if not np.isfinite(value) or value <= 0:
        return 0.02
    return float(np.clip(value / 1000.0, 0.001, 0.20))


def _add_model_feature_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Add optional composite model features while leaving core features unchanged."""
    out = df.copy()
    z_cols = [
        "ret_1d_z",
        "avg_to_close_5d_z",
        "imbalance_5d_z",
        "close_to_vwap_z",
        "amount_z20_z",
        "mom_5d_z",
        "mom_20d_z",
        "mom_60d_z",
        "spread_5d_z",
        "vol_20d_z",
        "range_5d_z",
        "illiquidity_z",
        "close_position_5d_z",
        "return_efficiency_z",
        "amount_trend_5d_z",
        "range_expansion_20d_z",
        "vol_change_5_20_z",
        "reversal_liquidity_z",
    ]
    for col in z_cols:
        raw = col[:-2]
        if col not in out.columns and raw in out.columns:
            out[col] = _cs_zscore(out, raw)
        elif col not in out.columns:
            out[col] = 0.0

    out["model_reversal_pressure"] = (
        out["avg_to_close_5d_z"] - out["close_to_vwap_z"] - 0.5 * out["ret_1d_z"]
    )
    out["model_liquidity_quality"] = (
        out["imbalance_5d_z"] - out["spread_5d_z"] - out["illiquidity_z"]
    )
    out["model_trend_quality"] = (
        out["mom_20d_z"] + 0.5 * out["mom_60d_z"] - out["vol_20d_z"] - 0.5 * out["range_5d_z"]
    )
    out["model_volume_reversal"] = (
        out["amount_z20_z"] - out["ret_1d_z"] - out["close_to_vwap_z"]
    )
    out["model_risk_compression"] = (
        -out["vol_20d_z"] - out["range_5d_z"] - out["spread_5d_z"]
    )
    out["model_close_strength"] = (
        out["close_position_5d_z"] + 0.5 * out["return_efficiency_z"] - 0.5 * out["range_expansion_20d_z"]
    )
    out["model_liquidity_reversal"] = (
        out["reversal_liquidity_z"] + 0.5 * out["amount_trend_5d_z"] - 0.5 * out["vol_change_5_20_z"]
    )
    out["model_crowding_unwind"] = (
        out["amount_trend_5d_z"] + out["range_expansion_20d_z"] - 0.5 * out["mom_5d_z"]
    )
    for col in [
        "model_reversal_pressure",
        "model_liquidity_quality",
        "model_trend_quality",
        "model_volume_reversal",
        "model_risk_compression",
        "model_close_strength",
        "model_liquidity_reversal",
        "model_crowding_unwind",
    ]:
        out[col] = _cs_zscore(out, col)
    return out


def _model_feature_columns(df: pd.DataFrame) -> List[str]:
    core = [
        "ret_1d", "avg_to_close", "avg_to_close_5d", "order_imbalance",
        "imbalance_5d", "spread_pct", "spread_5d", "range_pct", "range_5d",
        "amount_z20", "mom_5d", "mom_20d", "mom_60d", "vol_20d",
        "close_to_vwap", "illiquidity",
    ]
    composite = [
        "rule_factor",
        "model_reversal_pressure",
        "model_liquidity_quality",
        "model_trend_quality",
        "model_volume_reversal",
        "model_risk_compression",
        "model_close_strength",
        "model_liquidity_reversal",
        "model_crowding_unwind",
    ]
    z_features = [
        "ret_1d_z",
        "avg_to_close_5d_z",
        "imbalance_5d_z",
        "close_to_vwap_z",
        "amount_z20_z",
        "mom_20d_z",
        "mom_60d_z",
        "spread_5d_z",
        "vol_20d_z",
        "range_5d_z",
        "illiquidity_z",
        "close_position_5d_z",
        "return_efficiency_z",
        "amount_trend_5d_z",
        "range_expansion_20d_z",
        "vol_change_5_20_z",
        "reversal_liquidity_z",
    ]
    feature_set = str(MODEL_FEATURE_SET or "core").strip().lower()
    if feature_set == "expanded":
        selected = core + composite + z_features
    elif feature_set == "composite":
        selected = core + composite
    else:
        selected = core
    return [col for col in selected if col in df.columns]


def _finalize_factor(df: pd.DataFrame, start_dt: pd.Timestamp, end_dt: pd.Timestamp) -> pd.DataFrame:
    out = df.copy()
    out["ai_available"] = out["ai_model_pred"].notna()
    out["factor"] = np.where(
        out["ai_available"],
        MODEL_WEIGHT * out["ai_model_pred"] + RULE_WEIGHT * out["rule_factor"],
        out["rule_factor"],
    )
    if COMMON_NEUTRALIZE:
        out["factor"] = _neutralize_common_exposures(out, "factor")
    out["factor"] = FACTOR_SIGN * out["factor"]
    out["factor"] = out["factor"].replace([np.inf, -np.inf], np.nan)
    out["factor"] = out.groupby("date")["factor"].transform(lambda s: s.fillna(s.median()))
    out["factor"] = out["factor"].fillna(0.0)
    out["factor"] = _cs_zscore(out, "factor")
    out["factor"] = _postprocess_factor(out, "factor")

    result = out.loc[out["date"].between(start_dt, end_dt), ["date", "instrument", "factor"]].copy()
    result["date"] = pd.to_datetime(result["date"]).dt.normalize()
    result["instrument"] = result["instrument"].astype(str)
    result["factor"] = pd.to_numeric(result["factor"], errors="coerce").fillna(0.0).astype(float)
    result = result.sort_values(["date", "instrument"]).reset_index(drop=True)

    # Competition hard requirement: exactly these three columns.
    return result[["date", "instrument", "factor"]]


def _postprocess_factor(df: pd.DataFrame, factor_col: str) -> pd.Series:
    """Apply final daily cross-sectional shape control to improve robustness."""
    mode = str(FACTOR_POSTPROCESS_MODE or "zscore").strip().lower()
    base = pd.to_numeric(df[factor_col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    work = df.assign(_factor_base=base)
    if mode == "rank":
        ranked = work.groupby("date", group_keys=False)["_factor_base"].apply(_rank_to_uniform_score)
        work = work.assign(_factor_ranked=ranked)
        return _cs_zscore(work, "_factor_ranked")
    if mode == "tanh":
        work = work.assign(_factor_tanh=np.tanh(work["_factor_base"] / 2.0))
        return _cs_zscore(work, "_factor_tanh")
    return _cs_zscore(work, "_factor_base")


def _rank_to_uniform_score(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    s = s.fillna(s.median()).fillna(0.0)
    if len(s) <= 1:
        return pd.Series(0.0, index=s.index)
    ranked = s.rank(method="average", pct=True)
    return ((ranked - 0.5) * 2.0).clip(-1.0, 1.0)


def _neutralize_common_exposures(df: pd.DataFrame, factor_col: str) -> pd.Series:
    """Daily residualization against common price-volume style exposures."""
    exposure_cols = [
        "ret_1d",
        "mom_5d",
        "mom_20d",
        "mom_60d",
        "vol_20d",
        "range_5d",
        "spread_5d",
        "amount_z20",
        "illiquidity",
    ]
    available = [col for col in exposure_cols if col in df.columns]
    if not available:
        return df[factor_col]

    def residualize(group: pd.DataFrame) -> pd.Series:
        y = pd.to_numeric(group[factor_col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        y = y.fillna(y.median()).fillna(0.0).astype(float)
        if len(group) < max(20, len(available) + 5):
            return y

        x = group[available].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
        x = x.apply(lambda s: s.fillna(s.median()), axis=0).fillna(0.0).astype(float)
        x = x.loc[:, x.std(ddof=0) > EPS]
        if x.empty:
            return y
        x = (x - x.mean()) / (x.std(ddof=0) + EPS)
        x_mat = np.column_stack([np.ones(len(x)), x.to_numpy(dtype=float)])
        y_vec = y.to_numpy(dtype=float)
        ridge = np.eye(x_mat.shape[1]) * float(COMMON_NEUTRALIZE_RIDGE)
        ridge[0, 0] = 0.0
        try:
            beta = np.linalg.solve(x_mat.T @ x_mat + ridge, x_mat.T @ y_vec)
            resid = y_vec - x_mat @ beta
        except Exception:
            return y
        blended = (1.0 - COMMON_NEUTRALIZE_BLEND_ORIGINAL) * resid + COMMON_NEUTRALIZE_BLEND_ORIGINAL * y_vec
        return pd.Series(blended, index=group.index)

    residual = df.groupby("date", group_keys=False).apply(residualize)
    return pd.to_numeric(residual, errors="coerce").reindex(df.index).fillna(df[factor_col])


def main(
    data_source: Any = DEFAULT_BAR_TABLE,
    start_datetime: Any = "2025-01-01 00:00:00",
    end_datetime: Any = "2025-12-31 23:59:59",
    **kwargs: Any,
) -> pd.DataFrame:
    """Competition entry point.

    Parameters
    ----------
    data_source:
        The BigQuant platform may pass the data source name. If empty, the official
        minute-bar table `bigalpha_2026_stock_bar1m` is used.
    start_datetime, end_datetime:
        Evaluation window. The function returns factors only inside this window,
        but queries an earlier lookback window for rolling features and AI training.

    Returns
    -------
    pandas.DataFrame with exactly `date`, `instrument`, `factor`.
    """
    # Accept possible alternative argument names used by platform wrappers.
    start_datetime = kwargs.get("start_date", kwargs.get("start", start_datetime))
    end_datetime = kwargs.get("end_date", kwargs.get("end", end_datetime))

    start_dt = _to_timestamp(start_datetime, "2025-01-01 00:00:00").normalize()
    end_dt = _to_timestamp(end_datetime, "2025-12-31 23:59:59").normalize()
    if end_dt < start_dt:
        raise ValueError("end_datetime must be later than start_datetime")

    bar_table = _resolve_bar_table(data_source)
    query_start = max(pd.Timestamp("2019-01-01"), start_dt - pd.Timedelta(days=LOOKBACK_DAYS))

    daily = _load_daily_features_from_dai(bar_table, query_start, end_dt)
    features = _build_daily_features(daily)
    features = _add_rule_factor(features)
    features = _add_walk_forward_ai_factor(features, start_dt, end_dt)
    result = _finalize_factor(features, start_dt, end_dt)

    # Defensive validation before returning to the evaluator.
    if list(result.columns) != ["date", "instrument", "factor"]:
        raise ValueError("Submission output must contain exactly date, instrument, factor")
    if result["factor"].isna().any():
        raise ValueError("Submission output contains missing factor values")
    return result
